# 04. Mô hình hóa (Modeling)
- **Phân lớp (Classification)**: Dự đoán Machine Failure, tập trung xử lý mất cân bằng dữ liệu (SMOTE).
- **Hồi quy chuỗi thời gian (Forecasting)**: Dự báo độ mòn dao cụ (Tool wear).

In [ ]:
import pandas as pd
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.append(project_root)

from src.models.supervised import MaintenancePredictor
from src.models.forecasting import ToolWearForecaster
from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner

df_cleaned = pd.read_csv('../data/processed/ai4i2020_processed.csv')

## 1. Phân lớp có giám sát (Dự đoán hỏng hóc)
Giải bài toán phân lớp mất cân bằng (Machine failure).

In [ ]:
predictor = MaintenancePredictor(config_path='../configs/params.yaml')
X_train_res, X_test_scl, y_train_res, y_test_cls = predictor.split_and_prepare_data(df_cleaned)

### Huấn luyện và Đánh giá (Random Forest & XGBoost)

In [ ]:
models = predictor.train_models(X_train_res, y_train_res)

print("--- Đánh giá Random Forest ---")
rf_eval = predictor.evaluate_model("RandomForest", X_test_scl, y_test_cls)

print("--- Đánh giá XGBoost ---")
xgb_eval = predictor.evaluate_model("XGBoost", X_test_scl, y_test_cls)

### Feature Importance

In [ ]:
feature_names = predictor.models["XGBoost"].feature_names_in_ if hasattr(predictor.models["XGBoost"], 'feature_names_in_') else X_train_res.columns
predictor.plot_feature_importance("XGBoost", feature_names)

## 2. Hồi quy Dự báo (Forecasting Tool Wear)
Sử dụng thời gian xấp xỉ từ `UDI`.

In [ ]:
loader = DataLoader(config_path='../configs/params.yaml')
df_raw = loader.load_data()
cleaner = DataCleaner(config_path='../configs/params.yaml')
df_raw_clean = cleaner.handle_missing_values(df_raw)

forecaster = ToolWearForecaster(lag_steps=1)
df_lagged = forecaster.create_lag_features(df_raw_clean)
X_train_ts, X_test_ts, y_train_ts, y_test_ts = forecaster.prepare_data_split(df_lagged, test_size=0.2)

### Huấn luyện Ridge và XGBoost Regressor cho Tool Wear

In [ ]:
forecast_results, preds = forecaster.train_and_evaluate(X_train_ts, y_train_ts, X_test_ts, y_test_ts)
forecaster.plot_predictions(y_test_ts, preds, num_samples=300)